# DineDash — Temporary Views and Tables


### customers_tmp_view

In [0]:
create or replace temp view customers_tmp_view as
select * from read_files('/Volumes/amey_capstone_7405611437946584/dinedash/dinedash_volume/dimensions/dim_customers.csv', format => 'csv', header => true, inferSchema => true);

### restaurants_tmp_view

In [0]:
create or replace temp view restaurants_tmp_view as
select * from read_files('/Volumes/amey_capstone_7405611437946584/dinedash/dinedash_volume/dimensions/dim_restaurants.csv', format => 'csv', header => true, inferSchema => true);

### locations_tmp_view

In [0]:
create or replace temp view locations_tmp_view as
select * from read_files('/Volumes/amey_capstone_7405611437946584/dinedash/dinedash_volume/dimensions/dim_locations.csv', format => 'csv', header => true, inferSchema => true);

### delivery_agents_tmp_view

In [0]:
create or replace temp view delivery_agents_tmp_view as
select * from read_files('/Volumes/amey_capstone_7405611437946584/dinedash/dinedash_volume/dimensions/dim_delivery_agents.csv', format => 'csv', header => true, inferSchema => true);

### menu_items_tmp_view

In [0]:
create or replace temp view menu_items_tmp_view as
select * from read_files('/Volumes/amey_capstone_7405611437946584/dinedash/dinedash_volume/dimensions/dim_menu_items.csv', format => 'csv', header => true, inferSchema => true);

In [0]:
show views;

namespace,viewName,isTemporary,isMaterialized,isMetric
,_sqldf,true,false,false
,customers_tmp_view,true,false,false
,delivery_agents_tmp_view,true,false,false
,locations_tmp_view,true,false,false
,menu_items_tmp_view,true,false,false
,restaurants_tmp_view,true,false,false


## bronze tables

### customers_bronze

In [0]:
create or replace table amey_capstone_7405611437946584.dinedash.customers_bronze as
select *, current_timestamp() as ingestion_time, input_file_name() as source_file
from customers_tmp_view;

num_affected_rows,num_inserted_rows


### restaurants_bronze

In [0]:
create or replace table amey_capstone_7405611437946584.dinedash.restaurants_bronze as
select *, current_timestamp() as ingestion_time, input_file_name() as source_file
from restaurants_tmp_view;

num_affected_rows,num_inserted_rows


### locations_bronze

In [0]:
create or replace table amey_capstone_7405611437946584.dinedash.locations_bronze as
select *, current_timestamp() as ingestion_time, input_file_name() as source_file
from locations_tmp_view;

num_affected_rows,num_inserted_rows


### delivery_agents_bronze

In [0]:
create or replace table amey_capstone_7405611437946584.dinedash.delivery_agents_bronze as
select *, current_timestamp() as ingestion_time, input_file_name() as source_file
from delivery_agents_tmp_view;

num_affected_rows,num_inserted_rows


### menu_items_bronze

In [0]:
create or replace table amey_capstone_7405611437946584.dinedash.menu_items_bronze as
select *, current_timestamp() as ingestion_time, input_file_name() as source_file
from menu_items_tmp_view;

num_affected_rows,num_inserted_rows


## filtered dimension tables

### customers_filtered

In [0]:
create or replace table amey_capstone_7405611437946584.dinedash.customers_filtered as
select *
from amey_capstone_7405611437946584.dinedash.customers_bronze
where customer_id is not null and location_id is not null and signup_date is not null and dob is not null and email is not null and name is not null;

num_affected_rows,num_inserted_rows


### restaurants_filtered

In [0]:
create or replace table amey_capstone_7405611437946584.dinedash.restaurants_filtered as
select *
from amey_capstone_7405611437946584.dinedash.restaurants_bronze
where restaurant_id is not null and name is not null and cuisines is not null and location_id is not null and rating is not null and delivery_fee is not null;

num_affected_rows,num_inserted_rows


### locations_filtered

In [0]:
create or replace table amey_capstone_7405611437946584.dinedash.locations_filtered as
select *
from amey_capstone_7405611437946584.dinedash.locations_bronze
where location_id is not null and area is not null and city is not null and state is not null and latitude is not null and longitude is not null;

num_affected_rows,num_inserted_rows


### delivery_agents_filtered

In [0]:
create or replace table amey_capstone_7405611437946584.dinedash.delivery_agents_filtered as
select *
from amey_capstone_7405611437946584.dinedash.delivery_agents_bronze
where agent_id is not null and name is not null and phone is not null and rating is not null;

num_affected_rows,num_inserted_rows


### menu_items_filtered

In [0]:
create or replace table amey_capstone_7405611437946584.dinedash.menu_items_filtered as
select *
from amey_capstone_7405611437946584.dinedash.menu_items_bronze
where item_id is not null and restaurant_id is not null and item_name is not null and price is not null and category is not null;

num_affected_rows,num_inserted_rows


# DineDash — Incremental Orders Loading


In [0]:
create table if not exists amey_capstone_7405611437946584.dinedash.orders_bronze
using delta
as
select *
from read_files('/Volumes/amey_capstone_7405611437946584/dinedash/dinedash_volume/stream_data/orders_2024_01.json', format => 'json')
where 1 = 0;

num_affected_rows,num_inserted_rows


In [0]:
copy into amey_capstone_7405611437946584.dinedash.orders_bronze
from '/Volumes/amey_capstone_7405611437946584/dinedash/dinedash_volume/stream_data'
fileformat = json;

num_affected_rows,num_inserted_rows,num_skipped_corrupt_files
11632,11632,0


In [0]:
select count(*) as total_count
from amey_capstone_7405611437946584.dinedash.orders_bronze;

total_count
11632


### add march and load incrementally

In [0]:
copy into amey_capstone_7405611437946584.dinedash.orders_bronze
from '/Volumes/amey_capstone_7405611437946584/dinedash/dinedash_volume/stream_data'
fileformat = json;

num_affected_rows,num_inserted_rows,num_skipped_corrupt_files
5019,5019,0


In [0]:
select count(*) as total_count from amey_capstone_7405611437946584.dinedash.orders_bronze;

total_count
16651
